Running importance analysis with Covariates (Hipster Dataset)
=====================================

This is an *VariantSpark* example notebook that demonstrates training a random forest model using genotypes and associated covariates.

In this example, we use the Hipster dataset, which consists of:
 * `hipster.vcf` - A VCF file containing genotype data
 * `hipster_labels_covariates.txt` - A CSV file with sample-level covariates (age, ancestry principal components, etc.)
 * `hipster_labels.txt` - A CSV with sample labels

This example focuses on demonstrating how VariantSpark handles covariate integration alongside genomic features. VariantSpark enables you to incorporate arbitrary covariates directly into the modelling pipeline, allowing for adjustment to confounders or inclusion of additional predictors during training.

Step 1: Create a spark session with VariantSpark jar attached

In [ ]:
import varspark as vs
from pyspark.sql import SparkSession
spark = vs.configure_spark(
    SparkSession.builder.config('spark.jars', vs.find_jar())
).getOrCreate()

Step 2: Create a `VarsparkContext` using `SparkSession` object (here injected as `spark`):

In [4]:
vc = vs.VarsparkContext(spark, silent=True)

Step 3: Define covariate types 
VariantSpark supports specifying type information for each covariate. Covariate types determine how the variable is encoded and interpreted by the model.
Supported types include:
 * `CONTINUOUS` - a real valued numeric variable (e.g age, ancestry PCs)
 * `DISCRETE` - an integer valued variable where differences between values are meaningful
 * `NOMINAL(order_count)` - a categorical variable with no inherent ordering (e.g. sex, cohort). Order count is an optional integer defining the number of classes 
 * `ORDINAL(order_count)` - a categorical variable with an inherent ordering (e.g. severity scores). Order count is an optional integer defining the number of classes


In [5]:
covtypes = {
    "age": "CONTINUOUS",
    "PC0": "CONTINUOUS",
    "PC1": "CONTINUOUS",
    "PC2": "CONTINUOUS",
    "sex": "NOMINAL(2)",
    "lifestyle": "NOMINAL(4)",
}

Step 4: Load the genotypes `gts`, covariates `covs` and labels `ls` from data files.

In [6]:
gts = vc.import_vcf('../../data/hipsterIndex/hipster.vcf.bgz')
covs = vc.import_covariates('../../data/hipsterIndex/hipster_labels_covariates.txt', covtypes)
ls = vc.load_label('../../data/hipsterIndex/hipster_labels.txt', 'label')

../../data/hipsterIndex/hipster.vcf.bgz is loading to spark RDD, isBGZFile: true


Step 5: Merge the genotype and covariate sets into a single feature set `fs`. 

In [7]:
fs = vc.union_features_and_covariates(gts, covs)

Step 6: Build and train a random forest model on the unioned dataset:

In [8]:
rf_model = vs.RandomForestModel(vc, mtry_fraction=0.10, min_node_size=5, max_depth=10, seed=13)
rf_model.fit_trees(fs, ls, n_trees=300, batch_size=50)

Step 6: Display the results: print OOB error and variable importances

In [12]:
print("OOB error: %s" % rf_model.oob_error())
ia = rf_model.importance_analysis()
ia.important_variables()

OOB error: 0.05071884984025559


,variable,importance
0,2_223034082_A_G,131.707969
1,7_17284577_T_C,71.833121
2,5_126626044_A_C,47.504367
3,5_126627875_T_G,39.413753
4,5_126630016_C_T,37.795110
5,5_126630948_T_A,34.419896
6,4_54511913_G_A,27.721428
7,4_54510874_A_G,26.518174
8,5_126629112_C_T,26.458133
9,4_54509759_G_C,25.777392


Step 7: Since covariates didn't appear in the top important features (default limit=10), create a dataframe of all feature importances and filter to covariates

In [16]:
importance_scores = ia.variable_importance()
covariate_col_names = ['age', 'PC0', 'PC1', 'PC2', 'sex', 'lifestyle']
covariate_importances = importance_scores[
    importance_scores['variant_id'].isin(covariate_col_names)
]
covariate_importances


,variant_id,importance,splitCount
1005,PC2,1.629573,95
2033,PC1,1.626372,94
2034,age,1.729254,101
2972,PC0,1.921888,115
9147,lifestyle,1.140834,60
14344,sex,1.492220,46
